In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder

from preparation import charger, preparer

SEUIL = 131.0

NUM = ["masse_ordma_min"]
POLY = ["puiss_max"]
CAT = ["typ_boite_nb_rapp", "energ"]
CBL = "co2_mixte"             # la cible

df = charger("data/vehicules.csv")
d = preparer(df).copy()

d["malus"] = d["co2_mixte"] > SEUIL

In [ ]:
@dataclass(frozen=True)
class MetriquesClf:
    """Performance d'une prédiction binaire sur le jeu de test."""
    exactitude: float
    exactitude_naive: float      # toujours prédire la classe majoritaire
    matrice: np.ndarray          # confusion_matrix(yte, pred)
    n_train: int
    n_test: int

    @property
    def gain_sur_naif(self) -> float:
        return self.exactitude - self.exactitude_naive

In [2]:
@dataclass(frozen=True)
class Metriques:
    """Performance mesurée sur le jeu de test, et sa référence naïve."""
 
    mae: float
    r2: float
    mae_naive: float
    n_train: int
    n_test: int
 
    @property
    def rapport_au_naif(self) -> float:
        """De combien le modèle fait mieux qu'une prédiction constante."""
        return self.mae_naive / self.mae
 
    def __str__(self) -> str:
        return (
            f"MAE {self.mae:.1f} g/km (naif {self.mae_naive:.1f}, "
            f"x{self.rapport_au_naif:.1f}) | R2 {self.r2:.4f} | "
            f"{self.n_train} train / {self.n_test} test"
        )

In [ ]:
def construire_pipeline(estimateur: BaseEstimator) -> Pipeline:
    prep = ColumnTransformer([
        ("num",  StandardScaler(), NUM),
        ("poly", Pipeline([("p", PolynomialFeatures(2, include_bias=False)),
                        ("s", StandardScaler())]), POLY),
        ("cat",  OneHotEncoder(drop="first", handle_unknown="ignore"), CAT),
    ])
    return Pipeline([("prep", prep), ("model", estimateur)])


clf = construire_pipeline(LogisticRegression(max_iter=1000))
reg = construire_pipeline(LinearRegression())

In [ ]:
def entrainer(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 0
) -> tuple[Pipeline, Metriques]:
    X, y_co2, y_cls = d[NUM + POLY + CAT], d[CBL], d["malus"]
    
    Xtr, Xte, co2tr, co2te, ytr, yte = train_test_split(
    X, y_co2, y_cls, test_size=test_size, random_state=random_state, stratify=y_cls)

    modele_clf = clf.fit(Xtr, ytr)
    modele_reg = reg.fit(Xtr, co2tr)

    pred_clf = modele_clf.predict(Xte)
    pred_reg = modele_reg.predict(Xte)

    naive_clf = np.full(yte.shape, ytr.mean())
    naive_reg = np.full(co2te.shape, co2tr.mean())

    metriques_clf = MetriquesClf(
        mae=mean_absolute_error(yte, pred_clf),
        r2=r2_score(yte, pred_clf),
        mae_naive=mean_absolute_error(yte, naive_clf),
        n_train=len(Xtr),
        n_test=len(Xte),
    )

    metriques_reg = Metriques(
        mae=mean_absolute_error(co2te, pred_reg),
        r2=r2_score(co2te, pred_reg),
        mae_naive=mean_absolute_error(co2te, naive_reg),
        n_train=len(Xtr),
        n_test=len(Xte),
    )

    return (modele_clf, metriques_clf), (modele_reg, metriques_reg)